# Movie Recommendation — Step 2: Subsample + simplest baseline

**Goal of this notebook:** build the absolute simplest model possible,
confirm it runs end-to-end, and check how wrong it is. We deliberately
start dumb. Every later model only earns its complexity if it beats
this number.

We're working with a subsample first because the full `train.csv` has
~10 million rows — too slow to iterate on while we're still getting
the basics right. Once this works, we scale up.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

DATA_DIR = "data"  
SUBSAMPLE_SIZE = 500_000

## Load and subsample

We take a random 500K-row slice of the 10M rows. Using `random_state=42`
means this exact same subsample will be reproducible every time we re-run
this cell — important so our results don't shift randomly between runs.

In [2]:
train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
print(f"Full training data: {train_full.shape}")

if len(train_full) > SUBSAMPLE_SIZE:
    train = train_full.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    train = train_full.copy()

print(f"Working subsample: {train.shape}")
train.head()

Full training data: (10000038, 4)
Working subsample: (500000, 4)


,userId,movieId,rating,timestamp
0,122380,31445,2.5,1159967140
1,22380,56775,4.0,1343936580
2,104339,356,2.5,1111529397
3,64877,6874,4.0,1513800297
4,63164,2762,5.0,1005315064


## The simplest possible baseline: just guess the average

This model has zero intelligence. For every single user-movie pair,
it predicts the same number: the average rating across all of `train.csv`.
It doesn't know anything about who the user is or what the movie is.

This sounds too simple to be useful — but it's the benchmark every
other model must beat. If your fancy SVD++ model can't beat "always
guess 3.5 stars," something is wrong with the fancy model, not the
baseline.

In [3]:
global_mean = train["rating"].mean()
print(f"Global mean rating: {global_mean:.4f}")

train["baseline_pred"] = global_mean
train[["userId", "movieId", "rating", "baseline_pred"]].head()

Global mean rating: 3.5328


,userId,movieId,rating,baseline_pred
0,122380,31445,2.5,3.532775
1,22380,56775,4.0,3.532775
2,104339,356,2.5,3.532775
3,64877,6874,4.0,3.532775
4,63164,2762,5.0,3.532775


## How wrong is this baseline?

RMSE (root mean squared error) tells us, roughly, how many stars off
our predictions are on average; with bigger mistakes penalized more
than small ones. 

In [4]:
rmse = np.sqrt(mean_squared_error(train["rating"], train["baseline_pred"]))
print(f"RMSE of 'always guess the average': {rmse:.4f}")

RMSE of 'always guess the average': 1.0591


## What's next

This RMSE number is our number to beat. In the next notebook step, we'll:

1. Apply the **hide-and-predict trick** to THIS baseline — hide some
   ratings, "predict" them with just the global mean, and confirm the
   RMSE holds up on data the model never touched. This is the simplest
   possible demonstration of the validation method, before we use it
   on anything fancier.
2. Then build a slightly smarter baseline (user bias + movie bias) and
   watch the RMSE improve.

We are NOT touching SVD++ or LightGBM yet — one step at a time.